In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp 

import qutip as qq

In [2]:
# Helper Function

# Easier using 2->3


# Convertion helper 
# 1-> wave function  
# 2-> theta,phi 
# 3-> n,n location for hamiltonian
def C12(k,size):
    x=(k)%size[0]
    y=(k)//size[0]
    return x,y

def C13(k,j,size):
    i=k
    return i,j

def C21(pos,size):
    k=pos[0]+pos[1]*size[0]
    return k

def C23(pos,j,size):
    i=C21(pos,size)
    return i,j

def C31(i,j=0):
    return i

def C32(pos,size):
    x,y=C12(pos[0],size)
    return x,y

# Reconstruct Helper
def R12(par,size):
    i=np.array(range(size[0]*size[1]))
    val=np.zeros([size[1],size[0]],dtype=complex)
    x,y=C12(i,size)
    val[y,x]=par[i]
    return val

def R21(par,size):
    ipos=np.zeros([2,size[0]*size[1]],dtype=int)
    val=np.zeros(size[0]*size[1],dtype=complex)
    for j in range(size[1]):
        for i in range(size[0]):
            ipos[0,j*size[0]+i]=i
            ipos[1,j*size[0]+i]=j
    k=C21(ipos,size)
    r=range(size[0]*size[1])
    val[k]=par[ipos[0,r],ipos[1,r]]
    return val


def FDiffTheta(H,pos,a,dt,j,size):
    b=a/(2*dt)

    if pos[1]!=0:
        H[j,C21(pos+[0,-1],size)]-=b
    else:
        H[j,C21([pos[0],size[1]-1],size)]-=b
    if pos[1]!=(size[1]-1):
        H[j,C21(pos+[0, 1],size)]+=b
    else:
        H[j,C21([pos[0],0],size)]+=b
    return H

def SDiffTheta(H,pos,a,dt,j,size):
    b=a/dt**2

    H[j,C21(pos,size)]-=2*b
    if pos[1]!=0:
        H[j,C21(pos+[0,-1],size)]+=b
    else:
        H[j,C21([pos[0],size[1]-1],size)]+=b
    if pos[1]!=(size[1]-1):
        H[j,C21(pos+[0, 1],size)]+=b
    else:
        H[j,C21([pos[0],0],size)]+=b
    return H

def SDiffPhi(H,pos,a,dt,j,size):
    b=a/dt**2

    H[j,C21(pos,size)]-=2*b
    if pos[0]!=0:
        H[j,C21(pos+[-1,0],size)]+=b
    if pos[0]!=(size[0]-1):
        H[j,C21(pos+[ 1,0],size)]+=b
    return H

In [3]:
# Functionized Main
def main(phiE,ng,size,sphi=4*np.pi,stheta=np.pi,Isincludestate=False,GetHamiltonian=False):
    N,M=size
    NM=N*M

    H=np.zeros([NM,NM],dtype=complex)
    phi=np.linspace(-sphi,sphi,N)
    theta=np.linspace(-stheta,stheta,M)
    dp=phi[1]-phi[0]
    dt=theta[1]-theta[0]

    a=1000000000
    alpha=1
    beta=1
    Ect=10*a
    Ecp=10*a
    Ej=10*a
    El=50*a/1000
    # Ect=7.7*a
    # Ecp=2.5*a
    # El=1.88*a
    # Ej=27.2*a

    count=0
    for j in range(M):
        for i in range(N):
            pos=np.array([i,j])
            H[count,C21(pos,size)]= +4*(Ect*ng**2)\
                                    +El*(phi[i]-np.pi*phiE)**2\
                                    -2*Ej*np.cos(phi[i])*np.cos(theta[j])
            H=SDiffPhi(H,pos,-4*Ecp,dp,count,size)
            H=FDiffTheta(H,pos,1j*8*Ect*ng,dt,count,size)
            H=SDiffTheta(H,pos,-4*Ect,dt,count,size)
            count+=1
    if GetHamiltonian:
        return H
    # Solve using linalg normal
    if Isincludestate:
        a,b=np.linalg.eig(H)
        n=np.argsort(np.absolute(a))
        asort=a[n]
        wf=b[:,n]
        return asort,wf
    a=np.linalg.eigvals(H)
    n=np.argsort(np.absolute(a))
    asort=a[n]
    return asort
    
    # Solve using qutip
    # QH = qq.Qobj(H)
    # asort = QH.eigenenergies()
    # asort,wf = QH.eigenstates()


In [ ]:
# Functionized Main
def mainfz(phiE,ng,size,sphi=4*np.pi,stheta=np.pi,Isincludestate=False):
    N,M=size
    NM=N*M
    st=6

    H=np.zeros([NM,NM],dtype=complex)
    phi=np.linspace(-sphi,sphi,N)
    theta=np.linspace(-stheta,stheta,M)
    dp=phi[1]-phi[0]
    dt=theta[1]-theta[0]

    a=1000000000
    alpha=1
    beta=1
    Ect=10*a
    Ecp=10*a
    Ej=10*a
    El=50*a/1000
    # Ect=7.7*a
    # Ecp=2.5*a
    # El=1.88*a
    # Ej=27.2*a

    count=0
    for j in range(M):
        for i in range(N):
            pos=np.array([i,j])
            H[count,C21(pos,size)]= +4*(Ect*ng**2)\
                                    +El*(phi[i]-np.pi*phiE)**2\
                                    -2*Ej*np.cos(phi[i])*np.cos(theta[j])
            H=SDiffPhi(H,pos,-4*Ecp,dp,count,size)
            H=FDiffTheta(H,pos,1j*8*Ect*ng,dt,count,size)
            H=SDiffTheta(H,pos,-4*Ect,dt,count,size)
            count+=1

    # Solve using linalg normal
    if Isincludestate:
        a,b=sp.sparse.linalg.eigs(H,st,return_eigenvectors=True,which="SM")
        n=np.argsort(np.absolute(a))
        asort=a
        wf=np.transpose(b)
        return asort,wf
    a=sp.sparse.linalg.eigs(H,st,return_eigenvectors=False,which="SM")
    n=np.argsort(np.absolute(a))
    asort=a[n]
    return asort
    
    # Solve using qutip
    # QH = qq.Qobj(H)
    # asort = QH.eigenenergies()
    # asort,wf = QH.eigenstates()
#a=mainfz(0,0,[80,40])
#a

In [ ]:
# Testing
# %matplotlib qt

n=7
size=[40,20]
Enum=5
En=np.zeros([n,Enum])
sSweep=0.5
phiE=np.linspace(-sSweep,sSweep,n)
for i in range(n):
    En[i]=np.absolute(main(phiE[i],0.,size)[:Enum])

plt.plot(phiE,En[:,:6])





# sN=int(N/4)
# sM=int(M/2)
# new=np.transpose(R12(np.abs(wf[:,1]),size).real)
# new=np.transpose(R12(np.absolute(b[0]),size).real)
# new=R12(np.abs(wf[:,1]),size).real
# plt.imshow(new[20:60,20:],cmap='hot', interpolation='nearest')
# plt.imshow(new,cmap='hot', interpolation='nearest')
# plt.show()
# asort[:20].real
# R12(H[NM-1,:],size)
# n=10
# m=20
# test1=np.zeros([n*m])
# test2=np.zeros([m,n,2])
# test3=np.zeros([n*m,n*m,2])
# for i in range(n*m):
#     test1[i]=i
# for i in range(n):
#     for j in range(m):
#         test2[j,i,0]=i
#         test2[j,i,1]=j
# for j in range(n*m):
#     for i in range(n*m):
#         test3[j,i,0]=i
#         test3[j,i,1]=j



In [ ]:
# Testing Out
import ipywidgets as widgets
size=[5,8]
N,M=size
NM=N*M
H=main(0,0,size,GetHamiltonian=True)
np.set_printoptions(threshold=99)
np.set_printoptions(linewidth=1600)
dump=np.zeros([NM,M,N],dtype=complex)
for i in range(NM):
    dump[i]=R12(H[i],size)

x = widgets.IntSlider(description='x',max=N-1)
y = widgets.IntSlider(description='y',max=M-1)
def f(x, y):
    print(dump[C21([x,y],size)])

out = widgets.interactive_output(f, {'x': x, 'y': y},)

widgets.VBox([widgets.HBox([x, y]), out],layout=widgets.Layout(width='1400px'))



In [ ]:
size=[40,20]
E,wf=main(0,0,size,stheta=2*np.pi,Isincludestate=True)
im=np.abs(R12(wf[:,0],size))
# plt.imshow(np.transpose(im),extent=[-8*np.pi,8*np.pi,-np.pi,np.pi])
bound=[2,4]
plt.imshow(np.transpose(im),extent=[-bound[0],bound[0],-bound[1],bound[1]])

plt.show()
E[:6]